In [ ]:
# ==========================================
# BLOCK 1: ENVIRONMENT SETUP & PERSISTENCE
# ==========================================
# @title ⚙️ 1. Setup Environment & Google Drive
# @markdown Connects to Google Drive, sets up paths, and installs dependencies silently.

DRIVE_WORKSPACE_NAME = "AutoScribe_Workspace" # @param {type:"string"}
SHARED_DRIVE_NAME = "" # @param {type:"string"}
# @markdown *(Leave SHARED_DRIVE_NAME empty if you're using your personal MyDrive)*

import os
import sys
import time
from google.colab import drive
from IPython.display import clear_output, display
from IPython.utils import capture
import ipywidgets as widgets

# --- UI Helper Function ---
def inf(msg, style, wdth):
    inf_btn = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth))
    display(inf_btn)

print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- Determine Base Path (Shared vs Personal) ---
if SHARED_DRIVE_NAME != "" and os.path.exists(f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}"):
    root_path = f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}"
    print(f"Using Shared Drive: {SHARED_DRIVE_NAME}")
else:
    root_path = "/content/drive/MyDrive"

workspace_name = DRIVE_WORKSPACE_NAME.strip() or "AutoScribe_Workspace"
BASE_DIR = f"{root_path}/{workspace_name}"

# --- Set Up Directories ---
CACHE_DIR = os.path.join(BASE_DIR, "cache")
PIP_CACHE = os.path.join(CACHE_DIR, "pip_packages")
HF_CACHE = os.path.join(CACHE_DIR, "hf_models")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
TEMP_DIR = "/content/temp_media"
MARKER_FILE = os.path.join(CACHE_DIR, ".setup_complete")

for directory in [PIP_CACHE, HF_CACHE, OUTPUT_DIR, TEMP_DIR]:
    os.makedirs(directory, exist_ok=True)

# Point HuggingFace and Python to our Drive cache
os.environ["HF_HOME"] = HF_CACHE
sys.path.insert(0, PIP_CACHE)

# --- Silent & Fast Installation ---
if not os.path.exists(MARKER_FILE):
    print("First time setup: Installing dependencies to Drive. This will take ~3 minutes...")
    # capture.capture_output() swallows all red text, warnings, and clutter from pip
    with capture.capture_output() as cap:
        !pip install -q --target=$PIP_CACHE yt-dlp faster-whisper ffmpeg-python transformers accelerate torch torchvision
        
        with open(MARKER_FILE, 'w') as f:
            f.write("Setup complete.")

# --- Clean up terminal and show success button ---
clear_output()
inf('\u2714 Workspace Ready & Dependencies Loaded', 'success', '350px')

In [ ]:
# ==========================================
# BLOCK 2: INGESTION & ROUTING (yt-dlp, ffmpeg & Local Files)
# ==========================================
# @title 📥 2. Source Configuration & Routing
# @markdown Select your media source and AI fallback options:

SOURCE_TYPE = "URL" # @param ["URL", "Google_Drive", "Local_Upload"]
MEDIA_INPUT = "https://youtube.com/playlist?list=..." # @param {type:"string"}
WHISPER_MODE = "Auto" # @param ["On", "Off", "Auto"]
VISION_FALLBACK = True # @param {type:"boolean"}

import yt_dlp
import subprocess
import os
import shutil
from google.colab import files

routing_queue = []

def analyze_and_route(file_path, title, video_id):
    print(f"Analyzing audio levels via ffmpeg for: {title}...")
    command = ["ffmpeg", "-i", file_path, "-af", "silencedetect=noise=-50dB:d=2", "-f", "null", "-"]
    requires_vision = False
    
    try:
        result = subprocess.run(command, stderr=subprocess.PIPE, text=True)
        is_silent = "silence_start" in result.stderr or "Audio:" not in result.stderr
        
        if WHISPER_MODE == "Auto":
            if is_silent and VISION_FALLBACK:
                requires_vision = True
                print("[ROUTING]: Silence detected. Tagged for Vision Processing.")
            else:
                print("[ROUTING]: Audio intact. Tagged for Audio Processing (Whisper).")
        elif WHISPER_MODE == "Off" and VISION_FALLBACK:
            requires_vision = True
            print("[ROUTING]: Whisper forced OFF. Tagged for Vision Processing.")
            
    except Exception as e:
        print(f"Error during audio analysis: {e}. Fallback to Vision.")
        requires_vision = True

    routing_queue.append({
        'id': video_id, 'title': title, 'path': file_path, 'use_vision': requires_vision
    })

print(f"Processing source type: {SOURCE_TYPE}")

if SOURCE_TYPE == "Local_Upload":
    print("Please upload your media file(s)...")
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest_path = os.path.join(TEMP_DIR, filename)
        shutil.move(filename, dest_path)
        analyze_and_route(dest_path, filename, filename.split('.')[0])

elif SOURCE_TYPE == "Google_Drive":
    if os.path.exists(MEDIA_INPUT):
        filename = os.path.basename(MEDIA_INPUT)
        dest_path = os.path.join(TEMP_DIR, filename)
        shutil.copy2(MEDIA_INPUT, dest_path)
        analyze_and_route(dest_path, filename, filename.split('.')[0])
    else:
        print("❌ Error: File not found on Google Drive.")

elif SOURCE_TYPE == "URL":
    ydl_opts = {'format': 'bestaudio/best', 'outtmpl': f'{TEMP_DIR}/%(id)s.%(ext)s', 'quiet': True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(MEDIA_INPUT, download=False)
        entries = info['entries'] if 'entries' in info else [info]
        
        for entry in entries:
            v_id = entry['id']
            title = entry.get('title', v_id)
            print(f"\n--- Downloading: {title} ---")
            dl_info = ydl.extract_info(entry['url'], download=True)
            local_file = ydl.prepare_filename(dl_info)
            analyze_and_route(local_file, title, v_id)

print(f"✅ Block 2 Complete. {len(routing_queue)} item(s) in queue.")

In [ ]:
# ==========================================
# BLOCK 3: AI PROCESSING (Whisper & Moondream2)
# ==========================================
# @title 🧠 3. Execute AI Transcription & Vision Analysis
# @markdown This block processes the queue sequentially, managing GPU memory automatically.

import gc
import torch
import os
import subprocess
from faster_whisper import WhisperModel
from PIL import Image

# Dictionary to store final outputs
transcription_results = []

# --- Split Queue ---
audio_tasks = [t for t in routing_queue if not t['use_vision']]
vision_tasks = [t for t in routing_queue if t['use_vision']]

# ------------------------------------------
# STAGE 1: AUDIO PROCESSING (Whisper)
# ------------------------------------------
if audio_tasks:
    print(f"\n--- Loading Whisper Model (large-v3) ---")
    # ctranslate2 optimization for lower VRAM usage
    whisper_model = WhisperModel("large-v3", device="cuda", compute_type="float16", download_root=HF_CACHE)
    
    for task in audio_tasks:
        print(f"🎙️ Transcribing Audio: {task['title']}")
        try:
            segments, info = whisper_model.transcribe(task['path'], beam_size=5, language="sv", condition_on_previous_text=False)
            text_output = []
            for segment in segments:
                # Format timestamps [MM:SS]
                start_m, start_s = divmod(int(segment.start), 60)
                text_output.append(f"[{start_m:02d}:{start_s:02d}] {segment.text.strip()}")
            
            transcription_results.append({
                'title': task['title'],
                'content': "\n".join(text_output),
                'type': 'Audio Transcription'
            })
        except Exception as e:
            print(f"❌ Error transcribing {task['title']}: {e}")

    # VRAM Cleanup
    print("Flushing Whisper from GPU memory...")
    del whisper_model
    gc.collect()
    torch.cuda.empty_cache()

# ------------------------------------------
# STAGE 2: VISION PROCESSING (Moondream2)
# ------------------------------------------
if vision_tasks:
    print(f"\n--- Loading Moondream2 Vision Model ---")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    model_id = "vikhyatk/moondream2"
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision="2024-04-02", cache_dir=HF_CACHE)
    moondream = AutoModelForCausalLM.from_pretrained(
        model_id, trust_remote_code=True, revision="2024-04-02", cache_dir=HF_CACHE
    ).to("cuda")
    moondream.eval()

    for task in vision_tasks:
        print(f"👁️ Processing Vision: {task['title']}")
        frame_dir = os.path.join(TEMP_DIR, f"frames_{task['id']}")
        os.makedirs(frame_dir, exist_ok=True)
        
        # Extract 1 frame every 30 seconds to avoid overloading the model
        subprocess.run([
            "ffmpeg", "-i", task['path'], "-vf", "fps=1/30", 
            f"{frame_dir}/frame_%04d.jpg", "-hide_banner", "-loglevel", "error"
        ])
        
        frames = sorted(os.listdir(frame_dir))
        text_output = []
        
        for i, frame_file in enumerate(frames):
            img_path = os.path.join(frame_dir, frame_file)
            try:
                image = Image.open(img_path)
                enc_image = moondream.encode_image(image)
                answer = moondream.answer_question(
                    enc_image, 
                    "Describe the key action, text, or visual information in this frame concisely.", 
                    tokenizer
                )
                
                # Calculate timestamp based on frame index (30 sec intervals)
                time_sec = i * 30
                start_m, start_s = divmod(time_sec, 60)
                text_output.append(f"[{start_m:02d}:{start_s:02d}] {answer}")
            except Exception as e:
                print(f"Skipping frame {frame_file}: {e}")
                
        transcription_results.append({
            'title': task['title'],
            'content': "\n".join(text_output),
            'type': 'Visual Analysis'
        })

    # VRAM Cleanup
    print("Flushing Moondream2 from GPU memory...")
    del moondream
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Block 3 Complete. Processed {len(transcription_results)} items.")

In [ ]:
# ==========================================
# BLOCK 4: FORMATTING, EXPORT & CLEANUP
# ==========================================
# @title 📝 4. Export to Markdown & Auto-Shutdown
# @markdown Formats the text for NotebookLM, saves to Drive, and safely disconnects the runtime.

import os
import shutil
from google.colab import runtime
from datetime import datetime

# --- Configuration ---
MAX_WORDS_PER_FILE = 400000  # Safe margin below NotebookLM's 500k limit
PROJECT_NAME = "AutoScribe_Export" # Default name if none provided

print(f"\n--- Formatting Outputs for NotebookLM ---")

# Ensure Output Directory exists with a timestamped subfolder
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
export_dir = os.path.join(OUTPUT_DIR, f"{timestamp}_{PROJECT_NAME}")
os.makedirs(export_dir, exist_ok=True)

current_word_count = 0
file_index = 1
output_content = f"# {PROJECT_NAME} - Part {file_index}\n\n"

def save_markdown(index, content):
    filepath = os.path.join(export_dir, f"{PROJECT_NAME}_Part_{index}.md")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📁 Saved: {filepath}")

# --- Stitching Logic ---
for result in transcription_results:
    title = result['title']
    content = result['content']
    source_type = result['type']
    
    # Format individual video/source block using English tags for consistency
    block_text = f"## --- START SOURCE: {title} ---\n"
    block_text += f"**Analysis Type:** {source_type}\n\n"
    block_text += f"{content}\n\n"
    block_text += f"## --- END SOURCE: {title} ---\n\n"
    
    words_in_block = len(block_text.split())
    
    # Check if adding this block exceeds the NotebookLM limit
    if current_word_count + words_in_block > MAX_WORDS_PER_FILE:
        # Save current file and start a new one
        save_markdown(file_index, output_content)
        file_index += 1
        output_content = f"# {PROJECT_NAME} - Part {file_index}\n\n"
        output_content += f"> *[CONTINUED FROM PART {file_index-1}]*\n\n"
        current_word_count = 0
        
    output_content += block_text
    current_word_count += words_in_block

# Save the final (or only) file
if current_word_count > 0:
    save_markdown(file_index, output_content)

# ------------------------------------------
# CLEANUP & SHUTDOWN
# ------------------------------------------
print("\n--- Cleaning up temporary files ---")
try:
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
        print("🗑️ Ephemeral media storage cleared.")
except Exception as e:
    print(f"⚠️ Warning during cleanup: {e}")

print("\n🎉 All processing complete. Files are ready in your Google Drive.")
print("Disconnecting runtime to save resources...")

# Force Colab to disconnect and delete the VM instance
runtime.unassign()